# 학습 선수 7명의 무릎 통과 전후 연속 관측

기존 원판·자세 모델로 두 수동 판독 전후에 TCN에 넣을 연속 좌표가 있는지 확인합니다. 창 길이는 두 판독 중 이른 프레임의 5장 전부터 늦은 프레임의 5장 후까지입니다. 프레임 번호는 0부터 셉니다.

In [ ]:
from pathlib import Path
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "wlcv/config.py").is_file())
sys.path.insert(0, str(root))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from wlcv.evaluation.train_knee_pass_coverage import PARENT, extract, measure, plot_window

## 1. 원본 추론 결과 열기

기록한 실행 폴더가 있으면 그때의 좌표와 판독을 함께 읽습니다. 새로 추론하려면 아래 값을 True로 바꾸세요. 원판은 베이스라인 best.pt, 자세는 원본 YOLO26n-pose이며 입력 크기는 둘 다 640입니다. 수행자 선택과 원판 시간 연결은 영상마다 0번부터 시작합니다.

In [ ]:
recompute = False
folder = PARENT / "20261126-010624-911763"
if not folder.is_dir() and PARENT.is_dir():
    runs = sorted(PARENT.glob("*/frames.csv"))
    if runs:
        folder = runs[-1].parent
if recompute or not folder.is_dir():
    folder = extract(device="mps")
print(folder)

## 2. 선수별 관측과 끊긴 구간

동시 관측은 같은 원본 프레임에서 원판 중심과 선택된 수행자의 가까운 무릎 좌표를 모두 얻은 경우입니다. 결측 보간은 사용하지 않습니다. '통과 포함 전체 길이'는 분석 창 밖까지 조사해 두 판독 범위를 전부 포함하는 연속 프레임 수입니다. 해당 구간이 없으면 0입니다.

In [ ]:
summary, segments, window = measure(folder, padding=5)
display(summary.rename(columns={"video":"영상", "first":"1차", "second":"2차", "total":"창 길이",
    "paired":"동시 관측", "longest":"창 안 최장", "full_start":"전체 연속 시작", "full_end":"전체 연속 끝",
    "full_length":"통과 포함 전체 길이", "window_ms":"창의 시간(ms)"})[
    ["영상","1차","2차","창 길이","창의 시간(ms)","동시 관측","창 안 최장","전체 연속 시작","전체 연속 끝","통과 포함 전체 길이"]].round(1))
display(segments.rename(columns={"video":"영상", "start":"시작", "end":"끝", "length":"장수", "state":"상태"}))
fig = plot_window(summary, window)
plt.show()

## 3. 원본에 두 좌표 겹쳐 보기

아래 영상과 프레임 번호를 위 표에서 골라 바꿀 수 있습니다. 금색 +는 원판 박스 중심, 청록 ×는 가까운 무릎입니다. 통과 근처에서 좌표가 붙는지 눈으로 확인하세요.

In [ ]:
from wlcv.config import VIDEOS
from wlcv.data.frames import read_rgb

video, frame = "a06_sn_h_01", 19
points = pd.read_csv(folder / "frames.csv").set_index(["video", "frame"]).loc[(video, frame), :]
fig, ax = plt.subplots(figsize=(4, 5), layout="constrained")
ax.imshow(read_rgb(VIDEOS / f"{video}.mp4", frame))
if pd.notna(points.plate_y):
    ax.plot(points.plate_x, points.plate_y, "+", color="gold", markersize=13)
if pd.notna(points.knee_y):
    ax.plot(points.knee_x, points.knee_y, "x", color="cyan", markersize=10)
ax.set_title(f"{video} · {frame}")
ax.axis("off")
print("사람 후보 수", points.people, "· 무릎 상태", points.knee_status)
plt.show()

## 이번 실행에서 확인한 범위

82장 중 40장에서 두 좌표가 함께 나왔습니다. 나머지 42장에서는 사람 후보가 여럿이어서 수행자 선택을 보류했습니다. 두 판독을 가로질러 전후 2장씩 이어진 선수는 b01·b04·c02, 5장씩 이어진 선수는 b04 한 명입니다. 잘 찍힌 구간만 남기면 특정 촬영 환경에 편중될 수 있어, 다음에는 수행자 선택 시 복구와 오연결을 함께 확인합니다.